### **Marginal Footprinting - File Interrogation**

**<span style="color: darkred;">This is notebook 1/3 of the Marginal Footprinting downstream analyses.</span>**


Marginal Footprints are generated using the _scripts/chrombpnet_pipeline/5_MarginalFootprints.sh_ script

The following script outlines:
- Making .tsv lists of TF motifs for input to run the _5_MarginalFootprints.sh_
- The outputs generated from the Marginal Footprints function, and any file manipulation necessary prior to plotting.

Input required: **.h5 file**

All marginal footprint files are available in:

_/scratch/prj/stem_cells_pituitary/Georgia/ChromBPNet/outputs/(cohort)/(name)/marginal*/*.h5_

In [3]:
# Libraries
import h5py
import os
import hdf5plugin
import numpy as np
import glob

### <div style = 'background-color:PapayaWhip'> **Convert JASPAR PFMs to motif sequences**</div>

The MarginalFootprints function requires a key-value input in .tsv format to compute the footprint. The key is the TF motif name, and the value is the raw nucleotide sequence of the motif itself. 

The JASPAR 2026 motif file doesn't have these raw nucleotide sequences, it only contains the position-frequency matrix (PFM) of each motif. 

**The following converts these PFMs to a consensus sequence:**




In [4]:
# Paths
jaspar_2026="/scratch/prj/stem_cells_pituitary/Georgia/genome/JASPAR_CORE_2026_non-redundant.meme"
output_tsv="/scratch/prj/stem_cells_pituitary/Georgia/ChromBPnet/data/motif_sequences.tsv"
nucleotides = ['A', 'C', 'G', 'T']

os.path.exists(jaspar_2026)

True

In [5]:
# Function for generating .tsv file with no thresholds on sequence

print("Begin parsing MEME file.")

with open(jaspar_2026, "r") as infile, open(output_tsv, "w") as outfile:
    current_id = None
    matrix_lines = []
    is_reading_matrix = False

    for line in infile:
        line = line.strip()
        
        # Identify the start of a motif
        if line.startswith("MOTIF"):
            # End sequence of previous motif
            if current_id and matrix_lines:
                consensus = ""
                for row in matrix_lines:
                    probs = [float(x) for x in row.split()]
                    max_idx = probs.index(max(probs))
                    consensus += nucleotides[max_idx]
                outfile.write(f"{current_id}\t{consensus}\n")
            
            # Begin sequence of new motif
            parts = line.split()
            current_id = parts[1]
            matrix_lines = []
            is_reading_matrix = False
            
        # Identify the letter-probability matrix within the motif information
        elif line.startswith("letter-probability matrix"):
            is_reading_matrix = True
            continue
            
        # Extract the letter-probability matrix
        elif is_reading_matrix:
            if not line or line.startswith("URL"):
                is_reading_matrix = False
                continue
            matrix_lines.append(line)

    # Last motif in the file
    if current_id and matrix_lines:
        consensus = ""
        for row in matrix_lines:
            probs = [float(x) for x in row.split()]
            max_idx = probs.index(max(probs))
            consensus += nucleotides[max_idx]
        outfile.write(f"{current_id}\t{consensus}\n")

print(f"Successfully created {output_tsv}")

Begin parsing MEME file.
Successfully created /scratch/prj/stem_cells_pituitary/Georgia/ChromBPnet/data/motif_sequences.tsv


In [7]:
# Check
!head -n 5 /scratch/prj/stem_cells_pituitary/Georgia/ChromBPnet/data/motif_sequences.tsv

MA0002.1	TATTGTGGTTA
MA0003.1	GCCCGGGGG
MA0004.1	CACGTG
MA0006.1	TGCGTG
MA0007.1	ATAAGAACACCCTGTACCCGCC


### <div style = 'background-color:PapayaWhip'> **Make list of TF motif sequences**</div>

The JASPAR database contains 2059 TF motif sequences, which is too many to run at once. 

Filtered .tsv lists of TF motifs can be made for footprint analysis. 

**The following section generates .tsv files for a select list of TFs or motifs**

In the CPA paper -> there are distinct grouping lineage markers (TFs) that can be associated with different cell fates. 

The following:

1. Takes the TF names from each group
2. Uses an MA_id:Name dictionary to align each TF with an MA_id
3. Generates a .tsv file in the same format as the _motif_sequences.tsv_ file 

These files are needed as input for the Marginal Footprinting command.

In [9]:
# Generate dictionary of TFs to MA ID

jaspar_2026="/scratch/prj/stem_cells_pituitary/Georgia/genome/JASPAR_CORE_2026_non-redundant.meme"

# Create a disctionary with key = ID and value = motif name
id_to_motif = {}

# Extract these names from the jaspar file
with open(jaspar_2026, 'r') as file:
    for line in file:
        if line.startswith("MOTIF"):
            parts = line.split()
            # parts[0] is the word MOTIF
            # parts[1] is the ID
            # parts[2] is the Motif name
            if len(parts) >= 3:
                id_to_motif[parts[1]] = parts[2]

print(f'Loaded {len(id_to_motif)} motif names for mapping')

Loaded 2059 motif names for mapping


In [10]:
# Check file 
print(dict(list(id_to_motif.items())[:5]))

{'MA0002.1': 'RUNX1', 'MA0003.1': 'TFAP2A', 'MA0004.1': 'Arnt', 'MA0006.1': 'Ahr::Arnt', 'MA0007.1': 'Ar'}


In [11]:
# List of target TFs
# Pulled from the Consensus Pituitary Atlas paper: Kover et al. (2026)

#grouping_1_down = ["Bhlha15", "Mitf", "Npas4", "Onecut2", "Pbx3", "Rorb", "Scrt1", "Zeb1"]

grouping_1_up = ["Ebf1", "Elf3", "Grhl2", "Hic2", "Hnf4g", "Irf1", "Irf9", "Klf10", "Klf11", 
                 "Klf2", "Klf3", "Klf4", "Klf5", "Klf6", "Nfia", "Nfib", "Nfix", "Nfkb1", "Prop1", 
                 "Rela", "Relb", "Rest", "Rfx4", "Runx1", "Six1", "Six2", "Six4", "Sox2", "Sox4", 
                 "Sox6", "Sox8", "Sox9", "Stat1", "Stat6", "Tcf7l2", "Tead2", "Tead3", "Tead4"]

#grouping_2_down = ["Etv1", "Hlf", "Klf5", "Rorc"]

grouping_2_up = ["Foxl2", "Foxo6", "Nhlh2", "Nr5a1"]

grouping_4_down = ["Bcl11b", "Cebpd", "Dbp", "Dmrta2", "Ebf1", "Esrrg", "Foxl2", "Hif1a", "Irf1", 
                   "Klf9", "Myb", "Neurod1", "Nfil3", "Npas4", "Nr3c1", "Nr3c2", "Pitx1", "Pknox2", 
                   "Pou6f2", "Runx2", "Rxra", "Smad3", "Tbx19", "Tcf7l2", "Tef", "Tfeb", "Tgif1", 
                   "Trps1", "Twist2"]

grouping_4_up = ["Ascl1", "Bhlha15", "Elf3", "Etv1", "Foxp2", "Hnf4g", "Jdp2", "Maf", "Mef2c", 
                 "Meox1", "Msx1", "Nfatc1", "Nfatc2", "Nfia", "Nfib", "Nfix", "Nr2f2", "Pax6", 
                 "Pax7", "Pbx3", "Pknox1", "Plagl1", "Prdm9", "Prrx1", "Sox11", "Sox13", "Sox2", 
                 "Sox4", "Sox5", "Sox9", "Srebf2", "Stat6", "Tbx15", "Tfap4", "Vax1"]

#grouping_6_down = ["Gata2", "Gli2", "Grhl1", "Hlf", "Lef1", "Nr3c1", "Nr3c2", "Rxrg", "Srebf2", 
#                   "Stat2", "Tcf7l2"]

grouping_6_up = ["Ascl1", "Creb3l1", "Esr1", "Jdp2", "Lhx3", "Msx1", "Nfia", "Nfix", "Pitx2", 
                 "Pou1f1", "Pou2f2", "Pou6f2"]

#grouping_7_down = ["Arid3b", "Ascl1", "Lhx3", "Mef2c", "Pitx1", "Pitx2"]

grouping_7_up = ["Atf3", "Bach2", "Cebpb", "Cebpd", "Egr1", "Egr2", "Elf3", "Foxo1", "Foxp1", "Gli2",
                 "Glis3", "Irf1", "Klf9", "Mlxipl", "Nfatc2", "Nr3c1", "Nr3c2", "Plagl1", "Rarb", "Rfx4", 
                 "Sox2", "Sox6", "Stat3", "Stat4", "Tfeb"]

#grouping_8_down = ["Bach1", "Bhlha15", "Bnc2", "Cebpb", "Creb3l1", "Creb5", "Ebf1", "Ehf", "Fos", "Fosl2", 
#                   "Glis3", "Hsf4", "Jun", "Junb", "Jund", "Klf11", "Klf4", "Mafb", "Meis2", "Mxi1", 
#                   "Myb", "Myc", "Mycn", "Nfix", "Nfkb2", "Pknox1", "Plagl1", "Sox6", "Stat5b", "Tfap4", 
#                   "Tfcp2l1", "Tgif2", "Zbtb12"]

grouping_8_up = ["Dbp", "Foxl2", "Foxp2", "Gata2", "Hlf", "Isl1", "Lef1", "Nr5a1", "Pitx1", "Rorb", "Shox2"]

In [12]:
# Combine all lists and remove duplicates using set
all_groups = sorted(list(set(
    grouping_1_up + grouping_2_up + grouping_4_down + grouping_4_up + 
    grouping_6_up + grouping_7_up + grouping_8_up
)))

In [13]:
# Other lists that were used to make .tsv files for footprinting analysis
age_dependent_tfs = ["E2f8", "Lef1", "Tead2"]
trajectory_tfs = ["Smad3", "Foxl2", "Nr5a1", "Nhlh2", "Gata2", "Foxo6", "Lef1"]
trajectory_tfs_2 = ["Six2", "Rfx4", "Tead2"]
sox_tfs = ["Sox2", "Sox3", "Sox4", "Sox6", "Sox8", "Sox9", "Pou5f1::Sox2", "POU2F1::SOX2"]
gh3_data_tfs = ["Sox2", "Tead1", "Pou1f1", "Gli2", "Nr3c1", "Esr1", "NFIA", "NFIB", "NFIC", "NFIX"]
pou_tfs = ["POU2F2", "POU2F3", "POU6F1", "POU4F2", "POU1F1", "POU2F1", "POU3F1", "POU3F2", "POU3F3", "POU3F4", "POU4F1", "POU4F3", "POU5F1B", "POU6F2", "POU5F1", "POU6F1", "POU2F1::SOX2"]

In [ ]:
# Paths used to generate the tsv files
motif_sequences = "/scratch/prj/stem_cells_pituitary/Georgia/ChromBPnet/data/motif_sequences.tsv"
motif_sequences_subset = "/scratch/prj/stem_cells_pituitary/Georgia/ChromBPnet/data/motif_sequences_subset.tsv"

grouping_1_up_sub = "/scratch/prj/stem_cells_pituitary/Georgia/ChromBPnet/data/motif_sequences_g1_up.tsv"
grouping_2_up_sub = "/scratch/prj/stem_cells_pituitary/Georgia/ChromBPnet/data/motif_sequences_g2_up.tsv"
grouping_4_up_sub = "/scratch/prj/stem_cells_pituitary/Georgia/ChromBPnet/data/motif_sequences_g4_up.tsv"
grouping_4_down_sub = "/scratch/prj/stem_cells_pituitary/Georgia/ChromBPnet/data/motif_sequences_g4_down.tsv"
grouping_6_up_sub = "/scratch/prj/stem_cells_pituitary/Georgia/ChromBPnet/data/motif_sequences_g6_up.tsv"
grouping_7_up_sub = "/scratch/prj/stem_cells_pituitary/Georgia/ChromBPnet/data/motif_sequences_g7_up.tsv"
grouping_8_up_sub = "/scratch/prj/stem_cells_pituitary/Georgia/ChromBPnet/data/motif_sequences_g8_up.tsv"

all_groups_sub = "/scratch/prj/stem_cells_pituitary/Georgia/ChromBPnet/data/motif_sequences_allgroups.tsv"
age_dep_sub = "/scratch/prj/stem_cells_pituitary/Georgia/ChromBPnet/data/motif_sequences_age_dependent.tsv"
trajects = "/scratch/prj/stem_cells_pituitary/Georgia/ChromBPnet/data/motif_sequences_trajectory.tsv"
trajects_2 = "/scratch/prj/stem_cells_pituitary/Georgia/ChromBPnet/data/motif_sequences_trajectory_2.tsv"
sox_sub = "/scratch/prj/stem_cells_pituitary/Georgia/ChromBPnet/data/motif_sequences_SOX.tsv"
gh3_data_tfs_sub = "/scratch/prj/stem_cells_pituitary/Georgia/ChromBPnet/data/motif_sequences_GH3_lines.tsv"
pou_tf_sub = "/scratch/prj/stem_cells_pituitary/Georgia/ChromBPnet/data/POU_line_sequences.tsv"

In [ ]:
# Align TFs with their associated MA IDs from the JASPAR file

target_tfs_upper = {tf.upper() for tf in pou_tfs}

target_ids = []
found_in_jaspar = set()

for ma_id, name in id_to_motif.items():
    if name.upper() in target_tfs_upper:
        target_ids.append(ma_id)
        found_in_jaspar.add(name.upper())

missing_tfs = [tf for tf in pou_tfs if tf.upper() not in found_in_jaspar]

print(f"Total Target TFs: {len(pou_tfs)}")
print(f"TFs found in JASPAR: {len(found_in_jaspar)}")
print(f"Total MA IDs mapped: {len(target_ids)}")
print(f"TFs NOT found: {len(missing_tfs)}")

In [ ]:
# Check the TFs were correctly aligned
mapping_data = []
for ma_id in target_ids:
    mapping_data.append({
        "MA_ID": ma_id,
        "TF_Name": id_to_motif[ma_id]
    })
    
df_mapping = pd.DataFrame(mapping_data).sort_values("TF_Name")

print(df_mapping.head(5))

In [ ]:
# Generate the subset file
target_ids_set = set(target_ids)
count = 0

with open(motif_sequences, 'r') as infile, open(pou_tf_sub, 'w') as outfile:
    for line in infile:
        parts = line.strip().split('\t')
        if parts and parts[0] in target_ids_set:
            outfile.write(line)
            count += 1

print(f"TF Motif Sequences in file: {count}")

In [ ]:
# Check file
! head -n 5 /scratch/prj/stem_cells_pituitary/Georgia/ChromBPnet/data/motif_sequences_subset.tsv

In [ ]:
# Split into files of size 50 to run faster
#! split -l 50 -d --additional-suffix=.tsv /scratch/prj/stem_cells_pituitary/Georgia/ChromBPnet/data/motif_sequences_subset.tsv motif_subset_part_

In [ ]:
# Link sequence with MA id for a file already generated
import csv

motif_file='/scratch/prj/stem_cells_pituitary/Georgia/ChromBPnet/data/motif_sequences.tsv'
prolactin_motifs='/scratch/prj/stem_cells_pituitary/Georgia/ChromBPnet/data/prolactin_motif_list.tsv'
prolactin_final='/scratch/prj/stem_cells_pituitary/Georgia/ChromBPnet/data/prolactin_motif_list_2.tsv'

# Motif Sequences as a dictionary
motif_sequences = {}
with open(motif_file, 'r') as f:
    for line in f:
        parts = line.strip().split('\t')
        if len(parts) == 2:
            motif_sequences[parts[0]] = parts[1]

# Add sequences to MA-id file
with open(prolactin_motifs, 'r') as infile, \
     open(prolactin_final, 'w') as outfile:
    
    for line in infile:
        motif_id = line.strip()
        if motif_id in motif_sequences:
            outfile.write(f"{motif_id}\t{motif_sequences[motif_id]}\n")
        else:
            outfile.write(f"{motif_id}\t.\n")

print("Done!)

### <div style = 'background-color:PapayaWhip'> **.h5 file contents**</div>

In [ ]:
# Path

# Example file regions
Lacto_mf="/scratch/prj/stem_cells_pituitary/Georgia/ChromBPnet/outputs/mouse/Lactotrophs/marginal_footprints/Lactotrophs_FINAL_TOTAL_footprints.h5"

# Look at the .h5 file contents
with h5py.File(Lacto_mf, 'r') as file:
    # 1. See top level
    #print("Keys:", list(file.keys()))
    
    # Pick a group and see what the contents are
    group_name = 'MA0112.1'
    print(f"Inside {group_name}:", list(file[group_name].keys()))

Inside MA0112.1: ['i0', 'i1']


In [6]:
# What are the i0 and i1 values?

with h5py.File(Lacto_mf, 'r') as file:
    seq_data = file['MA0112.1/i0']
    seq_data2 = file['MA0112.1/i1']
    
    print(f"Shape: {seq_data.shape}")
    print(f"Datatype: {seq_data.dtype}")
    print(f"Shape: {seq_data2.shape}")
    print(f"Datatype: {seq_data2.dtype}")   

Shape: (1000,)
Datatype: float32
Shape: (1,)
Datatype: float32


**i0 is the sequence: 1000bp around the motif.**

It's not the sequence itself, its the probability values. This is what's used for the plotting.

**i1 is the magnitude**

This is a singular value that is used to normalize the profile or indicate the total predicted occupancy across that motif across the sampled regions.

### <div style = 'background-color:PapayaWhip'> **Merge multiple .h5 files**</div>

This is necessary if multiple runs of _5_MarginalFootprints.sh_ was conducted with different lists of motifs. 

To plot multiple footprints from multiple runs, it is easiest to have all footprints contained within one .h5 file instead of across multiple. 

In [ ]:
files_list = glob.glob('/scratch/prj/stem_cells_pituitary/Georgia/ChromBPnet/outputs/mouse/Lactotrophs/marginal_footprints/*.h5')
output_file = '/scratch/prj/stem_cells_pituitary/Georgia/ChromBPnet/outputs/mouse/Lactotrophs/marginal_footprints/Lactotrophs_footprints_combined.h5'

seen_keys = set()

with h5py.File(output_file, 'w') as out_f:
    for fpath in files_list:
        with h5py.File(fpath, 'r') as in_f:
            for key in in_f.keys():
                if key not in seen_keys:
                    in_f.copy(key, out_f)
                    seen_keys.add(key)
                else:
                    print(f"Skipping duplicate: '{key}' from {fpath}")

print(f"Merged keys: {list(out_f_keys := h5py.File(output_file,'r').keys())}")

The PCA analysis followed the following tutorial:

https://www.datacamp.com/tutorial/principal-component-analysis-in-python

In [ ]:
# Dictionary for Motif ID to Name

jaspar_2026="/scratch/prj/stem_cells_pituitary/Georgia/genome/JASPAR_CORE_2026_non-redundant.meme"

# key = ID 
# value = motif name
id_to_motif = {}

# Extract names from the JASPAR file
with open(jaspar_2026, 'r') as file:
    for line in file:
        if line.startswith("MOTIF"):
            parts = line.split()
            # parts[0] is the word 'MOTIF'
            # parts[1] is the MA* ID
            # parts[2] is the motif name
            if len(parts) >= 3:
                id_to_motif[parts[1]] = parts[2]

print(f'Loaded {len(id_to_motif)} motif names for mapping')